In [31]:
import pandas as pd
import numpy as np
import os

In [32]:
folder = 'D:/Project 3/Ecommerce_data_0.3/'

In [33]:
# Load all files
orders = pd.read_csv(folder + 'olist_orders_dataset.csv')
order_items = pd.read_csv(folder + 'olist_order_items_dataset.csv')
payments = pd.read_csv(folder + 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(folder + 'olist_order_reviews_dataset.csv')
customers = pd.read_csv(folder + 'olist_customers_dataset.csv')
products = pd.read_csv(folder + 'olist_products_dataset.csv')
sellers = pd.read_csv(folder + 'olist_sellers_dataset.csv')
category_translation = pd.read_csv(folder + 'product_category_name_translation.csv')

In [34]:
# Quick shape check on all
for name, df in [('orders', orders), ('order_items', order_items), ('payments', payments),
                  ('reviews', reviews), ('customers', customers), ('products', products),
                  ('sellers', sellers), ('category_translation', category_translation)]:
    print(f"{name}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
customers: (99441, 5)
products: (32951, 9)
sellers: (3095, 4)
category_translation: (71, 2)


In [35]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [36]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [37]:
# Check if order_id is truly unique in orders (should be, it's the primary entity)
print("orders unique order_id:", orders['order_id'].nunique(), "vs total rows:", orders.shape[0])

# Check order_items — order_id repeats expected (multiple products per order)
print("order_items unique order_id:", order_items['order_id'].nunique(), "vs total rows:", order_items.shape[0])

# Check payments — does order_id repeat here too?
print("payments unique order_id:", payments['order_id'].nunique(), "vs total rows:", payments.shape[0])

# Check reviews
print("reviews unique order_id:", reviews['order_id'].nunique(), "vs total rows:", reviews.shape[0])

orders unique order_id: 99441 vs total rows: 99441
order_items unique order_id: 98666 vs total rows: 112650
payments unique order_id: 99440 vs total rows: 103886
reviews unique order_id: 98673 vs total rows: 99224


In [38]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [39]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

In [40]:
# Derived column: actual delivery delay (only meaningful for delivered orders)
orders['delivery_delay_days'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days

orders['delivery_delay_days'].describe()

count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64

In [41]:
orders['delivery_delay_days'] = orders['delivery_delay_days'].round(0).astype('Int64')

In [42]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10


In [43]:
import io
import psycopg2

# ============================================
# Step 1:
# ============================================
conn = psycopg2.connect(
    host="localhost",
    dbname="ecommerce_capstone",
    user="postgres",
    password="123456",
    port="5432"
)
cur = conn.cursor()

# ============================================
# Step 2: Create all 7 tables (Olist schema)
# ============================================
cur.execute("""
CREATE TABLE customers (
    customer_id           VARCHAR(50) PRIMARY KEY,
    customer_unique_id    VARCHAR(50),
    customer_city         VARCHAR(100),
    customer_state        VARCHAR(5)
);

CREATE TABLE sellers (
    seller_id       VARCHAR(50) PRIMARY KEY,
    seller_city     VARCHAR(100),
    seller_state    VARCHAR(5)
);

CREATE TABLE products (
    product_id                  VARCHAR(50) PRIMARY KEY,
    product_category_name       VARCHAR(100),
    product_category_name_en    VARCHAR(100)
);

CREATE TABLE orders (
    order_id                        VARCHAR(50) PRIMARY KEY,
    customer_id                     VARCHAR(50) REFERENCES customers(customer_id),
    order_status                    VARCHAR(20),
    order_purchase_timestamp        TIMESTAMP,
    order_approved_at               TIMESTAMP,
    order_delivered_carrier_date    TIMESTAMP,
    order_delivered_customer_date   TIMESTAMP,
    order_estimated_delivery_date   TIMESTAMP,
    delivery_delay_days             INT
);

CREATE TABLE order_items (
    order_id             VARCHAR(50) REFERENCES orders(order_id),
    order_item_id         INT,
    product_id            VARCHAR(50) REFERENCES products(product_id),
    seller_id              VARCHAR(50) REFERENCES sellers(seller_id),
    price                   NUMERIC(10,2),
    freight_value            NUMERIC(10,2),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE TABLE payments (
    order_id                VARCHAR(50) REFERENCES orders(order_id),
    payment_sequential       INT,
    payment_type              VARCHAR(30),
    payment_installments       INT,
    payment_value                NUMERIC(10,2),
    PRIMARY KEY (order_id, payment_sequential)
);

CREATE TABLE reviews (
    review_id             VARCHAR(50),
    order_id               VARCHAR(50) REFERENCES orders(order_id),
    review_score             INT,
    review_comment_title      TEXT,
    review_comment_message    TEXT,
    review_creation_date       TIMESTAMP,
    review_answer_timestamp     TIMESTAMP,
    PRIMARY KEY (review_id, order_id)
);

CREATE INDEX idx_orders_customer ON orders(customer_id);
CREATE INDEX idx_orders_date ON orders(order_purchase_timestamp);
CREATE INDEX idx_order_items_order ON order_items(order_id);
CREATE INDEX idx_order_items_product ON order_items(product_id);
CREATE INDEX idx_order_items_seller ON order_items(seller_id);
CREATE INDEX idx_payments_order ON payments(order_id);
CREATE INDEX idx_reviews_order ON reviews(order_id);
""")
conn.commit()
print("All 7 tables created successfully")

# ============================================
# Step 3: ONE generic load function (reusable for all tables)
# ============================================
def load_table(dataframe, table_name):
    buffer = io.StringIO()
    dataframe.to_csv(buffer, index=False, header=False)
    buffer.seek(0)
    cur.copy_expert(f"COPY {table_name} FROM STDIN WITH CSV", buffer)
    conn.commit()
    print(f"{table_name} loaded — {len(dataframe)} rows")

# ============================================
# Step 4: Load in correct FK order
# ============================================
# 1. Customers
# ============================================
customers_df = customers[['customer_id', 'customer_unique_id', 
                            'customer_city', 'customer_state']].drop_duplicates(subset='customer_id')

# ============================================
# 2. Sellers
# ============================================
sellers_df = sellers[['seller_id', 'seller_city', 'seller_state']].drop_duplicates(subset='seller_id')

# ============================================
# 3. Products (merge with category translation for English names)
# ============================================
products_df = products.merge(
    category_translation, 
    on='product_category_name', 
    how='left'
)[['product_id', 'product_category_name', 'product_category_name_english']]

products_df = products_df.rename(columns={'product_category_name_english': 'product_category_name_en'})
products_df = products_df.drop_duplicates(subset='product_id')

# ============================================
# 4. Orders (includes derived delivery_delay_days from earlier step)
# ============================================
orders_df = orders[['order_id', 'customer_id', 'order_status', 
                      'order_purchase_timestamp', 'order_approved_at',
                      'order_delivered_carrier_date', 'order_delivered_customer_date',
                      'order_estimated_delivery_date', 'delivery_delay_days']]

# ============================================
# 5. Order Items
# ============================================
order_items_df = order_items[['order_id', 'order_item_id', 'product_id', 
                                'seller_id', 'price', 'freight_value']]

# ============================================
# 6. Payments
# ============================================
payments_df = payments[['order_id', 'payment_sequential', 'payment_type',
                          'payment_installments', 'payment_value']]

# ============================================
# 7. Reviews
# ============================================
reviews_df = reviews[['review_id', 'order_id', 'review_score',
                        'review_comment_title', 'review_comment_message',
                        'review_creation_date', 'review_answer_timestamp']]

# ============================================
# Shape check — sanity verification
# ============================================
for name, d in [('customers', customers_df), ('sellers', sellers_df), 
                 ('products', products_df), ('orders', orders_df),
                 ('order_items', order_items_df), ('payments', payments_df),
                 ('reviews', reviews_df)]:
    print(f"{name}: {d.shape}")
# ============================================
load_table(customers_df, "customers")
load_table(sellers_df, "sellers")
load_table(products_df, "products")
load_table(orders_df, "orders")
load_table(order_items_df, "order_items")
load_table(payments_df, "payments")
load_table(reviews_df, "reviews")

cur.close()
conn.close()
print("Done — connection closed")

All 7 tables created successfully
customers: (99441, 4)
sellers: (3095, 3)
products: (32951, 3)
orders: (99441, 9)
order_items: (112650, 6)
payments: (103886, 5)
reviews: (99224, 7)
customers loaded — 99441 rows
sellers loaded — 3095 rows
products loaded — 32951 rows
orders loaded — 99441 rows
order_items loaded — 112650 rows
payments loaded — 103886 rows
reviews loaded — 99224 rows
Done — connection closed


In [44]:
customers_df.isnull().sum()

customer_id           0
customer_unique_id    0
customer_city         0
customer_state        0
dtype: int64

In [45]:
products_df.isnull().sum()

product_id                    0
product_category_name       610
product_category_name_en    623
dtype: int64

In [46]:
orders_df.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_delay_days              2965
dtype: int64

In [47]:
order_items_df.isnull().sum()

order_id         0
order_item_id    0
product_id       0
seller_id        0
price            0
freight_value    0
dtype: int64

In [48]:
payments_df.isnull().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [49]:
reviews_df.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [50]:

products_df['product_category_name_en'] = products_df['product_category_name_en'].fillna('unknown_category')


products_df['product_category_name_en'].isnull().sum()

np.int64(0)

In [51]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    dbname="ecommerce_capstone",
    user="postgres",
    password="123456",
    port="5432"
)
cur = conn.cursor()

In [52]:
# TRUNCATE All TABLES customers,sellers,products,orders,order_items,payments,reviews   CASCADE; 
# Then loaded again all tables

load_table(customers_df, "customers")
load_table(sellers_df, "sellers")
load_table(products_df, "products")
load_table(orders_df, "orders")
load_table(order_items_df, "order_items")
load_table(payments_df, "payments")
load_table(reviews_df, "reviews")

customers loaded — 99441 rows
sellers loaded — 3095 rows
products loaded — 32951 rows
orders loaded — 99441 rows
order_items loaded — 112650 rows
payments loaded — 103886 rows
reviews loaded — 99224 rows


In [53]:
import pandas as pd

conn = psycopg2.connect(
    host="localhost",
    dbname="ecommerce_capstone",
    user="postgres",
    password="123456",
    port="5432"
)

query = """
WITH payments_agg AS (
    SELECT order_id, SUM(payment_value) AS total_payment_value,
           STRING_AGG(DISTINCT payment_type, ', ') AS payment_types
    FROM payments
    GROUP BY order_id
),
reviews_agg AS (
    SELECT order_id, AVG(review_score) AS avg_review_score
    FROM reviews
    GROUP BY order_id
)
SELECT 
    o.order_id, o.customer_id, c.customer_unique_id, c.customer_city, c.customer_state,
    o.order_status, o.order_purchase_timestamp, o.order_delivered_customer_date,
    o.order_estimated_delivery_date, o.delivery_delay_days,
    CASE 
        WHEN o.delivery_delay_days IS NULL THEN 'Not Delivered'
        WHEN o.delivery_delay_days <= -7 THEN 'Very Early'
        WHEN o.delivery_delay_days < 0 THEN 'Early'
        WHEN o.delivery_delay_days = 0 THEN 'On Time'
        WHEN o.delivery_delay_days <= 7 THEN 'Late'
        ELSE 'Very Late'
    END AS delivery_bucket,
    p.product_category_name_en AS category,
    oi.price, oi.freight_value, s.seller_id, s.seller_state,
    pa.payment_types, pa.total_payment_value, ra.avg_review_score
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
JOIN products p ON oi.product_id = p.product_id
JOIN sellers s ON oi.seller_id = s.seller_id
LEFT JOIN payments_agg pa ON o.order_id = pa.order_id
LEFT JOIN reviews_agg ra ON o.order_id = ra.order_id;
"""

export_df = pd.read_sql(query, conn)
export_df.to_csv(r'D:\Project 3\Ecommerce_data_0.3\all_in_one_ecommerce_data_export.csv', index=False)
conn.close()

print("Exported:", export_df.shape)
print("Total revenue check:", export_df['price'].sum())

C:\Users\turbo\AppData\Local\Temp\ipykernel_8372\3901349002.py:47: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  export_df = pd.read_sql(query, conn)


Exported: (112650, 19)
Total revenue check: 13591643.7


In [9]:
export_df.isnull().sum()

order_id                            0
customer_id                         0
customer_unique_id                  0
customer_city                       0
customer_state                      0
order_status                        0
order_purchase_timestamp            0
order_delivered_customer_date    2588
order_estimated_delivery_date       0
delivery_delay_days              2588
delivery_bucket                     0
category                            0
price                               0
freight_value                       0
seller_id                           0
seller_state                        0
payment_type                        3
payment_value                       3
review_score                      978
dtype: int64

In [10]:
export_df['price'].sum()

np.float64(14273699.65)